# Advanced Assignment
In this part of the advanced-level exercises, we will explore how to use our saved model to make predictions on unseen data.

Please return a Jupyter notebook as a submission in Canvas, to make the grading easier for us.


**Authors**: 
- Ariful Islam Riane : 3981886
- Pedro Fernandez Tonso : 3983781
- Group Number : 19

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import NLTKWordTokenizer
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences

I0000 00:00:1790670565.973239 3245765 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790670565.973630 3245765 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790670566.008446 3245765 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1790670566.892015 3245765 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790670566.892250 3245765 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### Exercise 1
Refer to Keras again on how to load your saved model, [here](https://www.tensorflow.org/guide/keras/save_and_serialize).

In [2]:
model_dir = Path.cwd() / "lab-1-dt-adv" / "models"
model = load_model(model_dir / "pos_tagger.h5")
x_tokenizer = tokenizer_from_json((model_dir / "x_tokenizer.json").read_text())
y_tokenizer = tokenizer_from_json((model_dir / "y_tokenizer.json").read_text())
max_len = model.input_shape[1]
model.summary()

E0000 00:00:1790670567.276374 3245765 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 100, 64)        │       728,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 100, 64)        │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 100, 13)        │           845 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 754,703 (2.88 MB)

 Trainable params: 754,701 (2.88 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [3]:
Corpus=['The articles in this special section focus on using natural language generation techniques (NLG) and natural language processing (NLP) to build computational systems that generate reports and other kinds of text in human languages.',
        'NLG uses analytics, AI, and NLP to obtain relevant information about non-linguistic data and to generate textual summaries and explanations of these data which help people understand and benefit from them.',
        'In this regard, NLG is a research field that addresses the data-value chain by using natural language as a tool for bridging the gap between raw data and valuable information communicated to users in a comprehensible way, adapted to their information needs.']

a. We want to be able to tag the three sentences in the corpus above [source](https://ieeexplore.ieee.org/document/7983468).

Figure out the necessary steps that should precede passing the input into your **h5** model.


In [4]:
# The model was trained on PTB-tokenized sentences (punctuation split off, case-insensitive
# vocabulary), padded/truncated to max_len. We apply the same steps: tokenize the sentences into
# words, map them to integers with the training tokenizer (unknown words become <OOV>) and pad.
# convert_parentheses maps ( and ) to -LRB- and -RRB-, which is how the treebank writes them
word_tokenizer = NLTKWordTokenizer()
tokens = [word_tokenizer.tokenize(sentence, convert_parentheses=True) for sentence in Corpus]
sequences = x_tokenizer.texts_to_sequences(tokens)
X_new = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')
print(X_new.shape)

(3, 100)


b. Use the model to make the predictions on this dataset

In [5]:
probs = model.predict(X_new)
pred_ids = probs.argmax(axis=-1)
print(probs.shape, pred_ids.shape)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 631ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 652ms/step


(3, 100, 13) (3, 100)


E0000 00:00:1790670568.055821 3245765 util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


c. Tag the corpus above

In [6]:
# drop the padding positions and map the predicted indices back to tags
tags = [[y_tokenizer.index_word[i] for i in ids[:len(toks)]] for ids, toks in zip(pred_ids, tokens)]

for toks, tgs in zip(tokens, tags):
    print(list(zip(toks, tgs)), "\n")

[('The', 'DET'), ('articles', 'NOUN'), ('in', 'ADP'), ('this', 'DET'), ('special', 'ADJ'), ('section', 'NOUN'), ('focus', 'NOUN'), ('on', 'ADP'), ('using', 'VERB'), ('natural', 'ADJ'), ('language', 'NOUN'), ('generation', 'NOUN'), ('techniques', 'NOUN'), ('-LRB-', '.'), ('NLG', 'NOUN'), ('-RRB-', '.'), ('and', 'CONJ'), ('natural', 'ADJ'), ('language', 'NOUN'), ('processing', 'NOUN'), ('-LRB-', '.'), ('NLP', 'NOUN'), ('-RRB-', '.'), ('to', 'PRT'), ('build', 'VERB'), ('computational', 'ADJ'), ('systems', 'NOUN'), ('that', 'ADP'), ('generate', 'X'), ('reports', 'VERB'), ('and', 'CONJ'), ('other', 'ADJ'), ('kinds', 'NOUN'), ('of', 'ADP'), ('text', 'NOUN'), ('in', 'ADP'), ('human', 'NOUN'), ('languages', 'NOUN'), ('.', '.')] 

[('NLG', 'NOUN'), ('uses', 'NOUN'), ('analytics', 'VERB'), (',', '.'), ('AI', 'VERB'), (',', '.'), ('and', 'CONJ'), ('NLP', 'NOUN'), ('to', 'PRT'), ('obtain', 'VERB'), ('relevant', 'ADJ'), ('information', 'NOUN'), ('about', 'ADP'), ('non-linguistic', 'ADJ'), ('data', 

d. Save the corpus and the predictions on a file, and upload it with your submission on Canvas.

In [7]:
rows = [(s, tok, tag) for s, (toks, tgs) in enumerate(zip(tokens, tags)) for tok, tag in zip(toks, tgs)]
predictions = pd.DataFrame(rows, columns=["sentence_id", "token", "pos_tag"])
predictions.to_csv(Path.cwd() / "lab-1-dt-adv" / "predictions.csv", index=False)
predictions.head(10)

,sentence_id,token,pos_tag
0,0,The,DET
1,0,articles,NOUN
2,0,in,ADP
3,0,this,DET
4,0,special,ADJ
5,0,section,NOUN
6,0,focus,NOUN
7,0,on,ADP
8,0,using,VERB
9,0,natural,ADJ
